# Gold Weekly Response Engine - Quantitative Research Exploration

Welcome to the **Gold Weekly Response Engine** interactive research notebook.
This notebook demonstrates how to:
1. Query the point-in-time relational database (`database/gold_research.db`) and load master Parquet datasets.
2. Inspect pre-event market state reconstruction and multi-asset regimes (Real Yields, DXY, VIX, Gold Trend).
3. Analyze weekly gold response distributions across reference definitions (Ref A through E).
4. Evaluate macro surprise elasticity, directional asymmetry, and conditional 2D response matrices.
5. Examine non-announcement market shocks, CFTC COT positioning extremes, and ETF flows.

In [ ]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import plotly.io as pio
pio.renderers.default = 'notebook'

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.statistics.event_study import EventStudyEngine
from src.statistics.conditional_matrix import ConditionalMatrixEngine
from src.statistics.shock_analysis import MarketShockAnalyzer
from src.statistics.positioning_analysis import PositioningAnalyzer
from src.visualization.event_plots import EventPlotter
from src.visualization.regime_plots import RegimePlotter
from src.visualization.weekly_plots import WeeklyPlotter

## 1. Load Master Research Datasets

We load both the event-level dataset (`data/processed/events_master.parquet`) and the weekly master dataset (`data/weekly/gold_weekly_master.parquet`).

In [ ]:
events_df = pd.read_parquet('../data/processed/events_master.parquet')
weekly_df = pd.read_parquet('../data/weekly/gold_weekly_master.parquet')

print(f"Loaded {len(events_df):,} historical macroeconomic events.")
print(f"Loaded {len(weekly_df):,} discrete trading weeks (2010 to present).")
events_df[['event_type', 'timestamp', 'consensus_value', 'actual_value', 'surprise_zscore', 'regime_real_yield', 'regime_dxy', 'event_to_next_fri_return']].head(10)

## 2. Querying via SQLite Relational Database

All records are persisted in `database/gold_research.db` for SQL auditing and integration.

In [ ]:
conn = sqlite3.connect('../database/gold_research.db')
query = """
SELECT 
    event_type,
    COUNT(*) as total_events,
    ROUND(AVG(surprise_zscore), 2) as avg_zscore,
    ROUND(AVG(event_to_next_fri_return) * 100, 2) as avg_weekly_return_pct
FROM events_master
GROUP BY event_type
ORDER BY total_events DESC
"""
sql_summary = pd.read_sql_query(query, conn)
conn.close()
sql_summary

## 3. Macroeconomic Event Study Analysis

We calculate the response across all horizons (5m, 1h, 1d, and Next Friday close) using robust bootstrap statistics.

In [ ]:
study_results = EventStudyEngine.run_event_study_by_type(events_df)
display_cols = ['event_type', 'sample_size', 'event_to_next_fri_return_median', 'event_to_next_fri_return_mean', 'event_to_next_fri_return_pos_rate']
study_results[[c for c in display_cols if c in study_results.columns]]

## 4. Conditional 2D Response Matrices

Cross-tabulate CPI and NFP responses by Real Yield Regimes (falling, neutral, rising).

In [ ]:
cpi_matrix = ConditionalMatrixEngine.build_conditional_matrix(
    events_df,
    event_type='CPI',
    regime_col='regime_real_yield'
)
print("=== CPI Median Return Matrix by Real Yield Regime (% * 100) ===")
display(cpi_matrix['median_matrix'] * 100.0)
print("=== Sample Size (N) per Cell ===")
display(cpi_matrix['count_matrix'])

## 5. Non-Announcement Market Shocks

Examine gold's forward 1-week performance following large moves (>2σ in DXY, Real Yields, VIX, S&P 500, WTI, Gold).

In [ ]:
shocks_results = MarketShockAnalyzer.analyze_shocks(weekly_df)
shocks_results[['shock_name', 'n_shocks', 'fwd_median_return', 'fwd_mean_return', 'positive_rate_pct', 'p_value']]

## 6. CFTC COT Positioning Extremes & ETF Flows

Evaluate whether extreme positioning acts as momentum or contrarian signals.

In [ ]:
cot_results = PositioningAnalyzer.analyze_cot_positioning(weekly_df)
display(cot_results)

etf_results = PositioningAnalyzer.analyze_etf_flows(weekly_df)
display(etf_results)